In [ ]:
import sys
project_path = "/home/aj/redesigned-octo-couscous" 
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import torch
from tensor_model import load_tensor_gpt
from transformers import GPT2Tokenizer

REPOSITORY = "Elriggs/gpt2-bilinear-18l-9h-1152embd"
device = torch.device("cuda")

In [ ]:
model, _, _ = load_tensor_gpt(REPOSITORY, device)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# model editing

class ModelEditor:
    def __init__(self, model):
        self.model = model
        self.layers = model.transformer.h
        self.original_states = {}

    def _save_original(self, layer):
        if layer not in self.original_states:
            mlp = self.layers[layer].mlp
            self.original_states[layer] = {
                "left_weight": mlp.Left.weight.detach().clone(),
                "right_weight": mlp.Right.weight.detach().clone(),
                "left_bias": mlp.Left_bias.detach().clone(),
                "right_bias": mlp.Right_bias.detach().clone(),
            }

    @staticmethod
    def _ablate_input_direction(weight, direction):
        direction = direction.to(device=weight.device, dtype=weight.dtype)
        norm_squared = direction.square().sum()
        if norm_squared == 0:
            raise ValueError("cannot ablate a zero direction")
        weight.sub_(
            (weight @ direction).unsqueeze(1)
            * (direction / norm_squared).unsqueeze(0)
        )

    def steer(self, vec1, vec2, layer):
        self._save_original(layer)
        mlp = self.layers[layer].mlp
        with torch.no_grad():
            mlp.Left_bias.copy_(
                vec1.to(device=mlp.Left_bias.device, dtype=mlp.Left_bias.dtype)
            )
            mlp.Right_bias.copy_(
                vec2.to(device=mlp.Right_bias.device, dtype=mlp.Right_bias.dtype)
            )

    def ablate(self, vec1, vec2, layer):
        self._save_original(layer)
        mlp = self.layers[layer].mlp
        with torch.no_grad():
            self._ablate_input_direction(mlp.Left.weight, vec1)
            self._ablate_input_direction(mlp.Right.weight, vec2)

    def restore(self):
        with torch.no_grad():
            for layer, state in self.original_states.items():
                mlp = self.layers[layer].mlp
                mlp.Left.weight.copy_(state["left_weight"])
                mlp.Right.weight.copy_(state["right_weight"])
                mlp.Left_bias.copy_(state["left_bias"])
                mlp.Right_bias.copy_(state["right_bias"])
        self.original_states.clear()

In [ ]:
with open(project_path+"/sparse-asymmetric/bilinear_attention_pr_weights/seed_0/ov_0.03/L.pt", "rb") as f:
    L = torch.load(f)
with open(project_path+"/sparse-asymmetric/bilinear_attention_pr_weights/seed_0/ov_0.03/R.pt", "rb") as f:
    R = torch.load(f)
with open(project_path+"/sparse-asymmetric/bilinear_attention_pr_weights/seed_0/ov_0.03/U.pt", "rb") as f:
    U = torch.load(f)
editor = ModelEditor(model)

In [ ]:
with torch.no_grad():
    simv = (U.t() @ U)
    simv = simv[torch.triu(torch.ones_like(simv), diagonal=1).bool()]
import seaborn as sns
sns.displot(simv.cpu())

In [ ]:
# look at factors
import random

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from torch.nn import functional as F
from tqdm.notebook import tqdm


def sample_fineweb_token_batches(
    tokenizer,
    device,
    num_contexts=256,
    sequence_length=32,
    seed=1729,
    shuffle_buffer=10_000,
    batch_size=32,
):
    """Sample fixed-length FineWeb contexts and return equal-sized token batches."""
    stream = load_dataset(
        "HuggingFaceFW/fineweb",
        "sample-10BT",
        split="train",
        streaming=True,
    ).shuffle(seed=seed, buffer_size=shuffle_buffer)
    rng = random.Random(seed)
    contexts = []
    with tqdm(total=num_contexts) as pbar:
        for row in stream:
            token_ids = tokenizer.encode(
                row["text"],
                add_special_tokens=False,
                verbose=False,
            )
            if len(token_ids) < sequence_length:
                continue
            start = rng.randrange(len(token_ids) - sequence_length + 1)
            contexts.append(token_ids[start:start + sequence_length])
            if len(contexts) == num_contexts:
                break
            pbar.update()
    if len(contexts) != num_contexts:
        raise RuntimeError("FineWeb stream ended before enough contexts were sampled")
    tokens = torch.tensor(contexts, device=device)
    return list(tokens.split(batch_size))


def exact_mixed_logit_effects(
    model,
    tokenizer,
    token_batches,
    left,
    right,
    source_layer=8,
    step=0.1,
):
    """Measure the full-model L x R interaction with a four-corner difference."""
    source_mlp = model.transformer.h[source_layer].mlp
    left = left.to(source_mlp.Left_bias.device)
    right = right.to(source_mlp.Right_bias.device)
    original_left_bias = source_mlp.Left_bias.detach().clone()
    original_right_bias = source_mlp.Right_bias.detach().clone()

    def logits(left_offset, right_offset):
        with torch.no_grad():
            source_mlp.Left_bias.copy_(original_left_bias + left_offset)
            source_mlp.Right_bias.copy_(original_right_bias + right_offset)
            return torch.cat([
                model.logits(token_ids)[:, -1, :len(tokenizer)].cpu()
                for token_ids in token_batches
            ]).float()

    try:
        zero_left = torch.zeros_like(left)
        zero_right = torch.zeros_like(right)
        mixed = logits(zero_left, zero_right)
        mixed -= logits(step * left, zero_right)
        mixed -= logits(zero_left, step * right)
        mixed += logits(step * left, step * right)
    finally:
        with torch.no_grad():
            source_mlp.Left_bias.copy_(original_left_bias)
            source_mlp.Right_bias.copy_(original_right_bias)
    return mixed / (step ** 2)


def split_half_reliability(effects, repeats=20, seed=1729):
    """Cosine agreement between mean effects from repeated disjoint context halves."""
    generator = torch.Generator().manual_seed(seed)
    values = []
    half = len(effects) // 2
    for _ in range(repeats):
        order = torch.randperm(len(effects), generator=generator)
        first_mean = effects[order[:half]].mean(dim=0)
        second_mean = effects[order[half:2 * half]].mean(dim=0)
        values.append(float(F.cosine_similarity(first_mean, second_mean, dim=0)))
    values = torch.tensor(values)
    return float(values.mean()), float(values.std())


def behavior_consistency_metrics(
    effects,
    top_tokens=12,
    split_repeats=20,
    split_seed=1729,
):
    """Summarize effect strength and reproducibility across sampled contexts."""
    effects = effects.float()
    normalized = F.normalize(effects, dim=1)
    mean_direction = F.normalize(effects.mean(dim=0), dim=0)
    context_alignment = normalized @ mean_direction
    eigenvalues = torch.linalg.eigvalsh(effects @ effects.T).clamp_min(0)
    split_mean, split_std = split_half_reliability(
        effects,
        repeats=split_repeats,
        seed=split_seed,
    )

    mean_effect = effects.mean(dim=0)
    token_count = min(top_tokens, mean_effect.numel())
    token_indices = torch.cat((
        torch.topk(mean_effect, token_count, largest=False).indices,
        torch.topk(mean_effect, token_count).indices,
    ))
    sign_consistency = torch.stack([
        (effects[:, index].sign() == mean_effect[index].sign()).float().mean()
        for index in token_indices
    ])
    return {
        "mean_effect_rms": float(effects.square().mean(dim=1).sqrt().mean()),
        "mean_context_alignment": float(context_alignment.mean()),
        "median_context_alignment": float(context_alignment.median()),
        "split_half_reliability": split_mean,
        "split_half_reliability_std": split_std,
        "top_component_energy_fraction": float(
            eigenvalues[-1] / eigenvalues.sum().clamp_min(1e-30)
        ),
        "median_top_token_sign_consistency": float(sign_consistency.median()),
        "minimum_top_token_sign_consistency": float(sign_consistency.min()),
    }


def scan_behavior_consistency(
    model,
    tokenizer,
    left_factors,
    right_factors,
    factor_indices=None,
    source_layer=8,
    num_contexts=256,
    sequence_length=32,
    context_batch_size=32,
    sample_seed=1729,
    shuffle_buffer=10_000,
    step=0.1,
    top_tokens=12,
    split_repeats=20,
    minimum_effect_quantile=0.5,
    display_count=20,
    figure_size=(15, 5),
):
    """Rank factors by reproducible exact mixed-logit behavior on random text."""
    token_batches = sample_fineweb_token_batches(
        tokenizer=tokenizer,
        device=next(model.parameters()).device,
        num_contexts=num_contexts,
        sequence_length=sequence_length,
        seed=sample_seed,
        shuffle_buffer=shuffle_buffer,
        batch_size=context_batch_size,
    )
    factor_indices = (
        list(range(left_factors.shape[1]))
        if factor_indices is None
        else list(factor_indices)
    )
    if not factor_indices:
        raise ValueError("factor_indices must contain at least one factor")

    rows = []
    for position, feature_index in enumerate(factor_indices, 1):
        effects = exact_mixed_logit_effects(
            model=model,
            tokenizer=tokenizer,
            token_batches=token_batches,
            left=left_factors[:, feature_index],
            right=right_factors[:, feature_index],
            source_layer=source_layer,
            step=step,
        )
        rows.append({
            "factor": feature_index,
            **behavior_consistency_metrics(
                effects,
                top_tokens=top_tokens,
                split_repeats=split_repeats,
                split_seed=sample_seed + feature_index,
            ),
        })
        if position % 10 == 0 or position == len(factor_indices):
            print(f"Scanned {position}/{len(factor_indices)} factors", end="\r")

    results = pd.DataFrame(rows)
    effect_threshold = results["mean_effect_rms"].quantile(minimum_effect_quantile)
    ranked = results[
        results["mean_effect_rms"] >= effect_threshold
    ].sort_values(
        ["split_half_reliability", "top_component_energy_fraction"],
        ascending=False,
    )

    figure, ax = plt.subplots(1, 1, figsize=figure_size, constrained_layout=True)
    scatter = ax.scatter(
        results["mean_effect_rms"],
        results["split_half_reliability"],
        c=results["top_component_energy_fraction"],
        cmap="viridis",
        alpha=0.82,
    )
    ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("Mean exact mixed-logit RMS")
    ax.set_ylabel("Split-half cosine reliability")
    ax.set_title("Behavioral strength and reproducibility")
    figure.colorbar(scatter, ax=ax, label="Top-component energy fraction")
    for row in ranked.head(12).itertuples():
        ax.annotate(str(row.factor), (row.mean_effect_rms, row.split_half_reliability))
    figure.suptitle(
        f"Exact mixed-behavior reliability | {len(results)} factors | "
        f"{num_contexts} random FineWeb contexts",
        fontsize=14,
    )
    plt.show()

    columns = [
        "factor",
        "mean_effect_rms",
        "split_half_reliability",
        "split_half_reliability_std",
        "mean_context_alignment",
        "top_component_energy_fraction",
        "median_top_token_sign_consistency",
        "minimum_top_token_sign_consistency",
    ]
    display(ranked[columns].head(display_count))
    return {
        "results": results,
        "ranked": ranked,
        "effect_threshold": float(effect_threshold),
        "config": {
            "metric_version": 3,
            "source_layer": source_layer,
            "num_contexts": num_contexts,
            "sequence_length": sequence_length,
            "context_batch_size": context_batch_size,
            "sample_seed": sample_seed,
            "shuffle_buffer": shuffle_buffer,
            "step": step,
            "top_tokens": top_tokens,
            "split_repeats": split_repeats,
            "minimum_effect_quantile": minimum_effect_quantile,
        },
    }


RESCAN_FACTORS = True
SCAN_CONFIG = {
    "metric_version": 3,
    "source_layer": 8,
    "num_contexts": 256,
    "sequence_length": 32,
    "context_batch_size": 32,
    "sample_seed": 1729,
    "shuffle_buffer": 10_000,
    "step": 0.1,
    "top_tokens": 12,
    "split_repeats": 20,
    "minimum_effect_quantile": 0.5,
}

cached_behavior_scan = globals().get("behavior_scan")
if RESCAN_FACTORS or cached_behavior_scan is None or cached_behavior_scan.get("config") != SCAN_CONFIG:
    behavior_scan = scan_behavior_consistency(
        model=model,
        tokenizer=tokenizer,
        left_factors=L,
        right_factors=R,
        factor_indices=range(L.shape[1]),
        **{key: value for key, value in SCAN_CONFIG.items() if key != "metric_version"},
    )
else:
    behavior_scan = cached_behavior_scan

In [ ]:
FACTOR_INDEX = int(behavior_scan["ranked"].iloc[6]["factor"])
print(FACTOR_INDEX)

In [ ]:
# FineWeb path decomposition of the exact L x R effect on u . resid
from collections import defaultdict
from math import prod

from torch.func import jvp
from tensor_model import apply_rotary_emb

SOURCE_NEURONS = 6
CIRCUIT_PATHS = 12
PATH_CONTEXTS = 256


def rms(values):
    return values * torch.rsqrt(values.square().mean(-1, keepdim=True) + torch.finfo(values.dtype).eps)


def attention_pattern(attention, normed):
    batch, sequence, _ = normed.shape
    shape = (batch, sequence, attention.n_head, attention.head_dim)
    query = attention.c_q(normed).view(shape)
    key = attention.c_k(normed).view(shape)
    cosine, sine = attention.rotary(query)
    query = apply_rotary_emb(rms(query), cosine, sine).transpose(1, 2)
    key = apply_rotary_emb(rms(key), cosine, sine).transpose(1, 2)
    logits = query @ key.transpose(-1, -2) / attention.head_dim ** 0.5
    causal = torch.ones(sequence, sequence, device=normed.device, dtype=torch.bool).tril()
    return logits.masked_fill(~causal, float("-inf")).softmax(dim=-1)


def attention_head_outputs(attention, normed, first_values, pattern=None):
    """Per-head writes [batch, head, position, embedding]; a fixed pattern gives the OV-only map."""
    batch, sequence, width = normed.shape
    shape = (batch, sequence, attention.n_head, attention.head_dim)
    value = (
        (1 - attention.lamb) * attention.c_v(normed).view(shape)
        + attention.lamb * first_values.view(shape)
    )
    if pattern is None:
        pattern = attention_pattern(attention, normed)
    head_values = pattern @ value.transpose(1, 2)
    output_weight = attention.c_proj.weight.view(width, attention.n_head, attention.head_dim).to(normed.dtype)
    return torch.einsum("bhtd,chd->bhtc", head_values, output_weight)


def measure_factor_paths(
    factor_index,
    token_batches,
    source_layer=8,
    target_layer=12,
    source_neurons=6,
    circuit_paths=12,
    target_positions=3,
    step=0.1,
):
    """Split the exact mixed effect on u . resid[target_layer] (mean of the last target positions).

    exact = J q + H[p_L, p_R], where q is the L x R write of the source MLP and p_L, p_R are its
    first-order writes. J q = direct residual + single-hop heads/MLPs + multi-hop compositions.
    """
    blocks = model.transformer.h
    source_mlp = blocks[source_layer].mlp
    layers = list(range(source_layer + 1, target_layer))
    positions = slice(-target_positions, None)
    device = source_mlp.Down.weight.device
    left = L[:, factor_index].to(device, torch.float32)
    right = R[:, factor_index].to(device, torch.float32)
    target = U[:, factor_index].to(device, torch.float32)

    with torch.no_grad():
        residual_scale = {layer: float(blocks[layer].lambdas[0]) for layer in layers}
        before = {layer: prod(residual_scale[k] for k in layers if k <= layer) for layer in layers}
        after = {layer: prod(residual_scale[k] for k in layers if k > layer) for layer in layers}
        direct_gain = prod(residual_scale.values())

        left_hidden = source_mlp.Left.weight.float() @ left
        right_hidden = source_mlp.Right.weight.float() @ right
        source_response = left_hidden * right_hidden
        source_energy = source_response.square()
        energy_fraction = source_energy / source_energy.sum().clamp_min(1e-30)
        neuron_ids = torch.topk(source_energy, source_neurons).indices.tolist()
        down = source_mlp.Down.weight.float()
        writes = {neuron: source_response[neuron] * down[:, neuron] for neuron in neuron_ids}
        total_write = down @ source_response
        writes["rest"] = total_write - sum(writes.values())
        original_left_bias = source_mlp.Left_bias.detach().clone()
        original_right_bias = source_mlp.Right_bias.detach().clone()

    def project(delta):
        return delta[..., positions, :].mean(dim=-2) @ target

    records = defaultdict(list)
    rms_sums = defaultdict(float)
    token_count = 0
    reimplementation_error = 0.0

    for tokens in tqdm(token_batches, leave=False):
        with torch.no_grad():
            state = model.state_before_block(tokens, source_layer)
            initial_values = state["initial_values"].float()
            first_values = state["first_values"].float()
            block = blocks[source_layer]
            values = block.lambdas[0] * state["values"].float() + block.lambdas[1] * initial_values
            values = values + block.attn(rms(values), first_values)[0]
            normed = rms(values)
            left_active = source_mlp.left(normed)
            right_active = source_mlp.right(normed)
            clean_source = values + source_mlp.Down(left_active * right_active) + source_mlp.Down_bias
            left_linear = source_mlp.Down(left_hidden * right_active)
            right_linear = source_mlp.Down(left_active * right_hidden)

            attention_inputs, mlp_inputs, patterns = {}, {}, {}
            values = clean_source
            for layer in layers:
                block = blocks[layer]
                values = block.lambdas[0] * values + block.lambdas[1] * initial_values
                attention_inputs[layer] = values
                patterns[layer] = attention_pattern(block.attn, rms(values))
                values = values + attention_head_outputs(block.attn, rms(values), first_values, patterns[layer]).sum(1)
                mlp_inputs[layer] = values
                values = values + block.mlp(rms(values))
                rms_sums[("attn", layer)] += float(attention_inputs[layer].square().mean(-1).sqrt().sum())
                rms_sums[("mlp", layer)] += float(mlp_inputs[layer].square().mean(-1).sqrt().sum())
            token_count += tokens.numel()
            reference = model.state_before_block(tokens, target_layer)["values"].float()
            reimplementation_error = max(
                reimplementation_error,
                float((values - reference).abs().max() / reference.abs().max()),
            )

            def downstream(values):
                for layer in layers:
                    block = blocks[layer]
                    values = block.lambdas[0] * values + block.lambdas[1] * initial_values
                    values = values + attention_head_outputs(block.attn, rms(values), first_values).sum(1)
                    values = values + block.mlp(rms(values))
                return project(values)

            records["linear"].append(
                jvp(downstream, (clean_source,), (torch.zeros_like(clean_source) + total_write,))[1]
            )
            records["second_order"].append(jvp(
                lambda values: jvp(downstream, (values,), (left_linear,))[1],
                (clean_source,), (right_linear,),
            )[1])

            def target_residual(left_offset, right_offset):
                source_mlp.Left_bias.copy_(original_left_bias + left_offset)
                source_mlp.Right_bias.copy_(original_right_bias + right_offset)
                return project(model.state_before_block(tokens, target_layer)["values"].float())

            try:
                finite = (
                    target_residual(step * left, step * right)
                    - target_residual(step * left, 0 * right)
                    - target_residual(0 * left, step * right)
                    + target_residual(0 * left, 0 * right)
                )
            finally:
                source_mlp.Left_bias.copy_(original_left_bias)
                source_mlp.Right_bias.copy_(original_right_bias)
            records["finite_difference"].append(finite / step ** 2)

            for source, write in writes.items():
                tangent = torch.zeros_like(clean_source) + write
                records[("linear", source)].append(jvp(downstream, (clean_source,), (tangent,))[1])
                records[("direct", source, None, None)].append(
                    torch.full((len(tokens),), direct_gain * float(write @ target), device=device)
                )
                for layer in layers:
                    attention, mlp = blocks[layer].attn, blocks[layer].mlp
                    layer_tangent = before[layer] * tangent
                    full = jvp(
                        lambda v: attention_head_outputs(attention, rms(v), first_values),
                        (attention_inputs[layer],), (layer_tangent,),
                    )[1]
                    frozen = jvp(
                        lambda v: attention_head_outputs(attention, rms(v), first_values, patterns[layer]),
                        (attention_inputs[layer],), (layer_tangent,),
                    )[1]
                    mlp_tangent = jvp(lambda v: mlp(rms(v)), (mlp_inputs[layer],), (layer_tangent,))[1]
                    full_scores = after[layer] * project(full)
                    ov_scores = after[layer] * project(frozen)
                    for head in range(attention.n_head):
                        records[("attn", source, layer, head)].append(full_scores[:, head])
                        records[("ov", source, layer, head)].append(ov_scores[:, head])
                    records[("mlp", source, layer, None)].append(after[layer] * project(mlp_tangent))

    values = {key: torch.cat(chunks).float().cpu() for key, chunks in records.items()}
    exact = values["linear"] + values["second_order"]
    # U has no intrinsic sign; orient it so the mean exact effect is positive.
    orientation = 1.0 if float(exact.mean()) >= 0 else -1.0
    values = {key: orientation * value for key, value in values.items()}
    exact = orientation * exact
    exact_mean = float(exact.mean())
    context_count = len(exact)

    def keys(component, source=None):
        return [
            key for key in values
            if isinstance(key, tuple) and len(key) == 4 and key[0] == component
            and (source is None or key[1] == source)
        ]

    def total(component, source=None):
        return sum(values[key] for key in keys(component, source))

    path_rows = []
    for component in ("direct", "attn", "mlp"):
        for key in keys(component):
            _, source, layer, head = key
            path_rows.append({
                "source_neuron": source,
                "component": component,
                "layer": layer,
                "head": head,
                "mean": float(values[key].mean()),
                "sem": float(values[key].std() / context_count ** 0.5),
                "fraction_of_exact": float(values[key].mean()) / exact_mean,
                "ov_only_fraction": (
                    float(values[("ov", source, layer, head)].mean()) / exact_mean
                    if component == "attn" else float("nan")
                ),
            })
    for source in writes:
        composition = values[("linear", source)] - total("direct", source) - total("attn", source) - total("mlp", source)
        path_rows.append({
            "source_neuron": source,
            "component": "multi-hop",
            "mean": float(composition.mean()),
            "sem": float(composition.std() / context_count ** 0.5),
            "fraction_of_exact": float(composition.mean()) / exact_mean,
        })
    paths = pd.DataFrame(path_rows)
    paths = paths.reindex(paths["mean"].abs().sort_values(ascending=False).index).reset_index(drop=True)

    route_mask = paths["component"].isin(["direct", "attn", "mlp"]) & (paths["source_neuron"] != "rest")
    routes = paths[route_mask].head(circuit_paths).reset_index(drop=True)
    routes["cumulative_fraction"] = routes["fraction_of_exact"].cumsum()
    shown = sum(
        values[(row.component, row.source_neuron, row.layer if row.component != "direct" else None,
                row.head if row.component == "attn" else None)]
        for row in routes.itertuples()
    )

    single_heads, single_mlps, direct = total("attn"), total("mlp"), total("direct")
    summary_terms = {
        "exact mixed (nested JVP)": exact,
        "exact mixed (finite difference, model forward)": values["finite_difference"],
        "linear transport J q": values["linear"],
        "  direct residual": direct,
        "  single-hop heads": single_heads,
        "    of which OV only (frozen pattern)": total("ov"),
        "  single-hop MLPs": single_mlps,
        "  multi-hop compositions": values["linear"] - direct - single_heads - single_mlps,
        "downstream second order H[p_L, p_R]": values["second_order"],
        f"top {len(routes)} routes (diagram)": shown,
    }
    summary = pd.DataFrame([
        {
            "term": name,
            "mean": float(term.mean()),
            "sem": float(term.std() / context_count ** 0.5),
            "fraction_of_exact": float(term.mean()) / exact_mean,
            "context_corr_with_exact": float(torch.corrcoef(torch.stack([term, exact]))[0, 1]),
        }
        for name, term in summary_terms.items()
    ]).set_index("term")

    rms_table = pd.DataFrame([
        {
            "layer": layer,
            "attn_input_rms": rms_sums[("attn", layer)] / token_count,
            "mlp_input_rms": rms_sums[("mlp", layer)] / token_count,
            "residual_gain_before": before[layer],
            "residual_gain_after": after[layer],
        }
        for layer in layers
    ]).set_index("layer")

    static_rows = []
    with torch.no_grad():
        for neuron in neuron_ids:
            for layer in layers:
                attention = blocks[layer].attn
                value_scale = float(1 - attention.lamb)
                for head in range(attention.n_head):
                    rows = slice(head * attention.head_dim, (head + 1) * attention.head_dim)
                    head_write = attention.c_proj.weight[:, rows].float() @ (attention.c_v.weight[rows].float() @ writes[neuron])
                    raw = orientation * value_scale * float(target @ head_write)
                    static_rows.append({
                        "source_neuron": neuron,
                        "layer": layer,
                        "head": head,
                        "static_raw": raw,
                        "static_normalized": raw * before[layer] * after[layer] / rms_table.loc[layer, "attn_input_rms"],
                        "measured_ov_only": float(values[("ov", neuron, layer, head)].mean()),
                        "measured_full": float(values[("attn", neuron, layer, head)].mean()),
                    })
    static_table = pd.DataFrame(static_rows)
    static_check = pd.DataFrame({
        "pearson_vs_measured_ov_only": static_table[["static_raw", "static_normalized"]].corrwith(static_table["measured_ov_only"]),
        "pearson_vs_measured_full": static_table[["static_raw", "static_normalized"]].corrwith(static_table["measured_full"]),
        "sum_ratio_to_measured_ov_only": static_table[["static_raw", "static_normalized"]].sum() / static_table["measured_ov_only"].sum(),
    })

    source_table = pd.DataFrame([
        {
            "source_neuron": neuron,
            "mixed_response": float(source_response[neuron]),
            "energy_fraction": float(energy_fraction[neuron]),
            "left_response": float(left_hidden[neuron]),
            "right_response": float(right_hidden[neuron]),
        }
        for neuron in neuron_ids
    ])
    return {
        "factor": factor_index,
        "source_layer": source_layer,
        "target_layer": target_layer,
        "orientation": orientation,
        "source_neurons": source_table,
        "paths": paths,
        "routes": routes,
        "summary": summary,
        "rms": rms_table,
        "static_vs_measured": static_table,
    }


path_token_batches = sample_fineweb_token_batches(
    tokenizer=tokenizer,
    device=next(model.parameters()).device,
    num_contexts=PATH_CONTEXTS,
    sequence_length=SCAN_CONFIG["sequence_length"],
    seed=SCAN_CONFIG["sample_seed"],
    shuffle_buffer=SCAN_CONFIG["shuffle_buffer"],
    batch_size=SCAN_CONFIG["context_batch_size"],
)
factor_paths = measure_factor_paths(
    factor_index=FACTOR_INDEX,
    token_batches=path_token_batches,
    source_layer=SCAN_CONFIG["source_layer"],
    target_layer=12,
    source_neurons=SOURCE_NEURONS,
    circuit_paths=CIRCUIT_PATHS,
    step=SCAN_CONFIG["step"],
)


In [ ]:
# show circuit
from matplotlib.patches import Circle, FancyBboxPatch, Rectangle

def show_factor_circuit(factor_paths):
    """Draw the top FineWeb single-hop routes (direct, heads, MLPs); labels are % of the exact mixed effect on u."""
    factor_index = factor_paths["factor"]
    source_layer, target_layer = factor_paths["source_layer"], factor_paths["target_layer"]
    path_table = factor_paths["routes"]
    source_table = factor_paths["source_neurons"]
    source_table = source_table[source_table["source_neuron"].isin(path_table["source_neuron"])]
    attention_layers = list(range(source_layer + 1, target_layer))
    head_count = model.config.n_head

    path_blue, direct_purple, positive, negative = "#2f45a8", "#8a3fa8", "#c9432c", "#1f8fb0"
    faded, dark, lavender = "#d6d6d6", "#30343b", "#d3d7ef"
    node_radius = 0.12
    column_spacing = 1.1
    lane_spacing = 0.26
    lane_start = 0.55
    direct_offset = 0.15

    neuron_ids = source_table["source_neuron"].tolist()
    neuron_info = source_table.set_index("source_neuron")
    width = max(2.0 + (len(neuron_ids) - 1) * column_spacing, 0.7 * head_count)
    neuron_offset = (width - (len(neuron_ids) - 1) * column_spacing) / 2
    neuron_x = {neuron: neuron_offset + k * column_spacing for k, neuron in enumerate(neuron_ids)}
    left_branch = {neuron: x - 0.3 for neuron, x in neuron_x.items()}
    right_branch = {neuron: x + 0.3 for neuron, x in neuron_x.items()}
    left_margin, right_margin = -1.1, width + 0.9
    label_gutter = 1.0

    def mlp_position(layer_index):
        return width + 0.2 + 0.2 * (layer_index - attention_layers[0])

    energy_max = float(source_table["energy_fraction"].max())
    neuron_strength = {
        row.source_neuron: (row.energy_fraction / energy_max) ** 0.5
        for row in source_table.itertuples()
    }
    max_route = float(path_table["fraction_of_exact"].abs().max())
    direct_rows = path_table[path_table["component"] == "direct"]

    def route_strength(value):
        return (abs(value) / max_route) ** 0.5

    def section_rows(top_y, routes):
        return top_y + lane_start + len(routes) * lane_spacing + 0.12

    def layer_routes(component, layer_index):
        routes = path_table[(path_table["component"] == component) & (path_table["layer"] == layer_index)]
        return routes.assign(x=routes["source_neuron"].map(neuron_x)).sort_values(["x", "head"])

    input_y, left_rail_y, right_rail_y = 0.3, 1.0, 1.2
    branch_y, product_y = 1.65, 2.25
    # Only layers/sublayers that carry a shown route get a section.
    sections = []
    y = product_y + 0.8
    for layer_index in attention_layers:
        for component in ("attn", "mlp"):
            routes = layer_routes(component, layer_index)
            if len(routes):
                node_y = section_rows(y, routes)
                sections.append((component, layer_index, y, routes, node_y))
                y = node_y + 0.4
    node_row = {(component, layer_index): node_y for component, layer_index, _, _, node_y in sections}
    resid_y = y
    collector_y = resid_y + 0.25
    u_x, u_y = width / 2, collector_y + 0.5
    total_height = u_y + 0.3

    # Shift each attention row so its heads keep clear of trunks, direct wires and earlier active heads.
    head_spacing = width / head_count
    obstacles = list(neuron_x.values()) + [neuron_x[n] - direct_offset for n in direct_rows["source_neuron"]]
    head_shift = {}
    for component, layer_index, _, routes, _ in sections:
        if component != "attn":
            continue
        active_heads = set(routes["head"].astype(int))
        def clearance(shift):
            return min(abs(head_spacing * (head + shift) - x) for head in active_heads for x in obstacles)
        head_shift[layer_index] = max([0.1 + 0.05 * k for k in range(18)], key=clearance)
        obstacles += [head_spacing * (head + head_shift[layer_index]) for head in active_heads]

    def head_position(layer_index, head):
        return head_spacing * (int(head) + head_shift[layer_index])

    figure, axis = plt.subplots(
        figsize=(0.5 * (right_margin + label_gutter - left_margin), 0.5 * (total_height + 0.1)),
        constrained_layout=True,
    )

    def value_color(value):
        return positive if value > 0 else negative

    def format_fraction(value):
        return f"{100 * value:+.1f}%" if abs(value) < 0.1 else f"{100 * value:+.0f}%"

    def multiply_node(x, y):
        axis.add_patch(Circle((x, y), node_radius, facecolor="white", edgecolor=dark, linewidth=1.3, zorder=4))
        offset = node_radius * 0.6
        for sign in (-1, 1):
            axis.plot([x - offset, x + offset], [y - sign * offset, y + sign * offset], color=dark, linewidth=1.0, zorder=5)

    def wire(points, strength, color=path_blue):
        xs, ys = zip(*points)
        axis.plot(xs, ys, color=color, linewidth=0.7 + 3.0 * strength, alpha=0.35 + 0.6 * strength, zorder=2)

    def separator(y, label):
        axis.plot([left_margin, right_margin], [y, y], color=dark, linewidth=1.5, linestyle=(0, (2.5, 2.5)), zorder=0)
        axis.text(right_margin + 0.08, y, label, ha="left", va="center", fontsize=8)

    def norm_band(top_y, label):
        separator(top_y, label)
        axis.add_patch(Rectangle(
            (left_margin, top_y + 0.1), right_margin - left_margin, 0.24,
            facecolor="#f0f0f3", edgecolor="#a9a9a9", linewidth=0.8, zorder=0.5,
        ))
        axis.text(left_margin + 0.06, top_y + 0.22, "rmsnorm", ha="left", va="center", fontsize=6.5, color="#6b6b6b", zorder=0.6)

    def lanes(top_y, routes, stop_x, node_y):
        for lane, row in enumerate(routes.itertuples()):
            lane_y = top_y + lane_start + lane * lane_spacing
            score = float(row.fraction_of_exact)
            start_x = neuron_x[row.source_neuron]
            wire(
                [(start_x, product_y), (start_x, lane_y), (stop_x(row), lane_y), (stop_x(row), node_y)],
                route_strength(score),
            )
            axis.text(start_x + 0.05, lane_y - 0.03, format_fraction(score), ha="left", va="bottom", fontsize=5.5, color=value_color(score))

    def box(x, y, box_width, box_height, label, fontsize, linewidth):
        axis.add_patch(FancyBboxPatch(
            (x - box_width / 2, y - box_height / 2), box_width, box_height,
            boxstyle="square,pad=0", facecolor=lavender, edgecolor="black", linewidth=linewidth, zorder=3,
        ))
        axis.text(x, y, label, ha="center", va="center", fontsize=fontsize, family="monospace", zorder=4)

    left_box_x, right_box_x = width * 0.3, width * 0.7
    box(left_box_x, input_y, 1.2, 0.5, f"L{factor_index}", 10, 1.8)
    box(right_box_x, input_y, 1.2, 0.5, f"R{factor_index}", 10, 1.8)
    separator(0.75, f"{source_layer}.mlp")
    for box_x, rail_y, branch in (
        (left_box_x, left_rail_y, left_branch),
        (right_box_x, right_rail_y, right_branch),
    ):
        wire([(box_x, input_y + 0.25), (box_x, rail_y)], 1.0)
        wire([(min(box_x, *branch.values()), rail_y), (max(box_x, *branch.values()), rail_y)], 0.7)

    for neuron in neuron_ids:
        x = neuron_x[neuron]
        strength = neuron_strength[neuron]
        for branch_x, rail_y, value, label_offset in (
            (left_branch[neuron], left_rail_y, float(neuron_info.loc[neuron, "left_response"]), -1),
            (right_branch[neuron], right_rail_y, float(neuron_info.loc[neuron, "right_response"]), 1),
        ):
            wire([(branch_x, rail_y), (branch_x, branch_y)], strength)
            wire([(branch_x, branch_y), (branch_x, product_y), (x, product_y)], strength)
            multiply_node(branch_x, branch_y)
            axis.text(
                branch_x + 0.14 * label_offset,
                branch_y + 0.16 * label_offset,
                f"{value:.2g}",
                ha="left" if label_offset > 0 else "right",
                va="top" if label_offset > 0 else "bottom",
                fontsize=6,
                color=value_color(value),
            )
        multiply_node(x, product_y)
        energy = float(neuron_info.loc[neuron, "energy_fraction"])
        axis.text(x + 0.17, product_y + 0.15, f"N{neuron}\n{100 * energy:.1f}%", ha="left", va="top", fontsize=6)

    for component, layer_index, top_y, routes, node_y in sections:
        norm_band(top_y, f"{layer_index}.{component}")
        if component == "attn":
            for head in sorted(set(routes["head"].astype(int))):
                x = head_position(layer_index, head)
                multiply_node(x, node_y)
                axis.text(x - 0.15, node_y - 0.13, "V", ha="right", va="bottom", fontsize=6, color=dark)
                axis.text(x + 0.15, node_y + 0.13, f"H{head}", ha="left", va="top", fontsize=6, color=dark)
            lanes(top_y, routes, lambda row: head_position(layer_index, row.head), node_y)
        else:
            mlp_x = mlp_position(layer_index)
            multiply_node(mlp_x, node_y)
            axis.text(mlp_x + 0.15, node_y + 0.13, "mlp", ha="left", va="top", fontsize=6, color=dark)
            lanes(top_y, routes, lambda row: mlp_x, node_y)

    separator(resid_y, f"{target_layer}.resid")
    # Every route descends into the residual sum bar, which is then read out by u.
    collectors = []
    for (layer_index, head), group in path_table[path_table["component"] == "attn"].groupby(["layer", "head"]):
        x = head_position(layer_index, head)
        collectors.append(([(x, node_row[("attn", layer_index)])], group["fraction_of_exact"].abs().sum(), path_blue))
    for layer_index, group in path_table[path_table["component"] == "mlp"].groupby("layer"):
        x = mlp_position(layer_index)
        collectors.append(([(x, node_row[("mlp", layer_index)])], group["fraction_of_exact"].abs().sum(), path_blue))
    for row in direct_rows.itertuples():
        score = float(row.fraction_of_exact)
        x = neuron_x[row.source_neuron]
        direct_x = x - direct_offset
        collectors.append(([(x, product_y + node_radius), (direct_x, product_y + 0.3)], abs(score), direct_purple))
        axis.text(
            direct_x - 0.08, product_y + 0.3, f"direct\n{format_fraction(score)}",
            ha="right", va="top", fontsize=6, color=value_color(score),
        )
    collector_max = max(total for _, total, _ in collectors)
    for start, total, color in collectors:
        wire(start + [(start[-1][0], collector_y)], (total / collector_max) ** 0.5, color)
    bar_xs = [start[-1][0] for start, _, _ in collectors] + [u_x]
    axis.plot([min(bar_xs), max(bar_xs)], [collector_y, collector_y], color=dark, linewidth=2.3, solid_capstyle="round", zorder=3)
    axis.plot([u_x, u_x], [collector_y, u_y - 0.2], color=dark, linewidth=2.3, zorder=3)
    axis.text(max(bar_xs) + 0.08, collector_y, "Σ", ha="left", va="center", fontsize=9, color=dark)
    box(u_x, u_y, 1.2, 0.4, f"U{factor_index}", 10, 1.8)

    axis.set_xlim(left_margin - 0.05, right_margin + label_gutter)
    axis.set_ylim(total_height, -0.05)
    axis.set_aspect("equal")
    axis.set_axis_off()
    plt.show()
    return {"source_neurons": source_table, "routes": path_table}


factor_circuit = show_factor_circuit(factor_paths)


In [ ]:
# tokens
TOKENS_PER_DIRECTION = 12

def show_factor_token_effects(
    factor_index,
    tokens_per_direction=12,
    num_contexts=256,
    sequence_length=32,
    context_batch_size=32,
    sample_seed=1729,
    left_factors=None,
    right_factors=None,
    source_layer=None,
    step=None,
    shuffle_buffer=None,
):
    left_factors = L if left_factors is None else left_factors
    right_factors = R if right_factors is None else right_factors
    source_layer = SCAN_CONFIG["source_layer"] if source_layer is None else source_layer
    step = SCAN_CONFIG["step"] if step is None else step
    shuffle_buffer = SCAN_CONFIG["shuffle_buffer"] if shuffle_buffer is None else shuffle_buffer
    token_batches = sample_fineweb_token_batches(
        tokenizer=tokenizer,
        device=next(model.parameters()).device,
        num_contexts=num_contexts,
        sequence_length=sequence_length,
        seed=sample_seed,
        shuffle_buffer=shuffle_buffer,
        batch_size=context_batch_size,
    )
    effects = exact_mixed_logit_effects(
        model=model,
        tokenizer=tokenizer,
        token_batches=token_batches,
        left=left_factors[:, factor_index],
        right=right_factors[:, factor_index],
        source_layer=source_layer,
        step=step,
    )
    mean_effect = effects.mean(dim=0)
    standard_error = effects.std(dim=0) / len(effects) ** 0.5
    token_count = min(tokens_per_direction, mean_effect.numel())

    rows = []
    for direction, largest in (("promoted", True), ("suppressed", False)):
        indices = torch.topk(mean_effect, token_count, largest=largest).indices
        for rank, index in enumerate(indices, 1):
            rows.append({
                "direction": direction,
                "rank": rank,
                "token": repr(tokenizer.decode([int(index)])),
                "token_id": int(index),
                "mean_logit_effect": float(mean_effect[index]),
                "standard_error": float(standard_error[index]),
                "sign_consistency": float(
                    (effects[:, index].sign() == mean_effect[index].sign()).float().mean()
                ),
            })

    token_effects = pd.DataFrame(rows)
    plotted = pd.concat([
        token_effects[token_effects["direction"] == "suppressed"].iloc[::-1],
        token_effects[token_effects["direction"] == "promoted"],
    ])
    colors = plotted["direction"].map({"promoted": "#24796c", "suppressed": "#b34b42"})
    _, axis = plt.subplots(constrained_layout=True)
    axis.barh(
        plotted["token"],
        plotted["mean_logit_effect"],
        xerr=plotted["standard_error"],
        color=colors,
        alpha=0.9,
    )
    axis.axvline(0, color="#333333", linewidth=0.8)
    axis.set_xlabel("Mean exact mixed-logit effect")
    axis.set_ylabel("GPT-2 token")
    axis.set_title(
        f"Factor {factor_index}: token promotion and suppression"
    )
    plt.show()
    return token_effects


factor_token_effects = show_factor_token_effects(
    factor_index=FACTOR_INDEX,
    tokens_per_direction=TOKENS_PER_DIRECTION,
    num_contexts=SCAN_CONFIG["num_contexts"],
    sequence_length=SCAN_CONFIG["sequence_length"],
    context_batch_size=SCAN_CONFIG["context_batch_size"],
    sample_seed=SCAN_CONFIG["sample_seed"],
)

In [ ]:
# held-out end-of-model steering validation

STEERING_CONTEXTS = 256
STEERING_SEED = SCAN_CONFIG["sample_seed"] + 10_000


def validate_final_token_steering(
    factor_index,
    token_effects,
    num_contexts=256,
    sequence_length=32,
    context_batch_size=32,
    sample_seed=11_729,
    step=0.1,
    left_factors=None,
    right_factors=None,
    source_layer=None,
    shuffle_buffer=None,
):
    """Test discovered token effects on held-out final logits and probabilities."""
    left_factors = L if left_factors is None else left_factors
    right_factors = R if right_factors is None else right_factors
    source_layer = SCAN_CONFIG["source_layer"] if source_layer is None else source_layer
    shuffle_buffer = SCAN_CONFIG["shuffle_buffer"] if shuffle_buffer is None else shuffle_buffer
    token_batches = sample_fineweb_token_batches(
        tokenizer=tokenizer,
        device=next(model.parameters()).device,
        num_contexts=num_contexts,
        sequence_length=sequence_length,
        seed=sample_seed,
        shuffle_buffer=shuffle_buffer,
        batch_size=context_batch_size,
    )
    source_mlp = model.transformer.h[source_layer].mlp
    left = left_factors[:, factor_index].to(source_mlp.Left_bias.device, source_mlp.Left_bias.dtype)
    right = right_factors[:, factor_index].to(source_mlp.Right_bias.device, source_mlp.Right_bias.dtype)
    original_left_bias = source_mlp.Left_bias.detach().clone()
    original_right_bias = source_mlp.Right_bias.detach().clone()
    selected_ids = torch.tensor(
        token_effects["token_id"].tolist(),
        device=source_mlp.Left_bias.device,
        dtype=torch.long,
    )

    def run_corner(left_offset, right_offset):
        with torch.no_grad():
            source_mlp.Left_bias.copy_(original_left_bias + left_offset)
            source_mlp.Right_bias.copy_(original_right_bias + right_offset)
            selected_logits, selected_probabilities = [], []
            for token_ids in token_batches:
                final_logits = model.logits(token_ids)[:, -1, :len(tokenizer)]
                selected_logits.append(final_logits.index_select(1, selected_ids).cpu())
                selected_probabilities.append(
                    final_logits.softmax(dim=-1).index_select(1, selected_ids).cpu()
                )
        return torch.cat(selected_logits).float(), torch.cat(selected_probabilities).float()

    zero_left, zero_right = torch.zeros_like(left), torch.zeros_like(right)
    try:
        corners = {
            "00": run_corner(zero_left, zero_right),
            "10": run_corner(step * left, zero_right),
            "01": run_corner(zero_left, step * right),
            "11": run_corner(step * left, step * right),
        }
    finally:
        with torch.no_grad():
            source_mlp.Left_bias.copy_(original_left_bias)
            source_mlp.Right_bias.copy_(original_right_bias)

    baseline_logits, baseline_probabilities = corners["00"]
    paired_logits = corners["11"][0] - baseline_logits
    paired_probabilities = corners["11"][1] - baseline_probabilities
    mixed_logits = corners["11"][0] - corners["10"][0] - corners["01"][0] + corners["00"][0]
    mixed_probabilities = corners["11"][1] - corners["10"][1] - corners["01"][1] + corners["00"][1]
    one_sided_logits = paired_logits - mixed_logits
    one_sided_probabilities = paired_probabilities - mixed_probabilities
    interaction_logits = mixed_logits / step ** 2
    interaction_probabilities = mixed_probabilities / step ** 2

    results = token_effects.copy().reset_index(drop=True)
    expected_sign = results["direction"].map({"promoted": 1.0, "suppressed": -1.0})
    measurements = {
        "paired_logit_delta": paired_logits,
        "paired_probability_delta": paired_probabilities,
        "mixed_logit_contribution": mixed_logits,
        "mixed_probability_contribution": mixed_probabilities,
        "one_sided_logit_contribution": one_sided_logits,
        "one_sided_probability_contribution": one_sided_probabilities,
        "interaction_logit_effect": interaction_logits,
        "interaction_probability_effect": interaction_probabilities,
    }
    for name, values in measurements.items():
        results[name] = values.mean(dim=0).numpy()
        results[f"{name}_sem"] = (values.std(dim=0) / len(values) ** 0.5).numpy()
    results["baseline_probability"] = baseline_probabilities.mean(dim=0).numpy()
    results["paired_probability_delta_ppm"] = 1e6 * results["paired_probability_delta"]
    results["expected_sign"] = expected_sign

    predicted = torch.tensor(results["mean_logit_effect"].to_numpy())
    expected = torch.tensor(expected_sign.to_numpy())
    summary_rows = []
    for name, values in (
        ("paired steer: final logits", paired_logits),
        ("paired steer: final probabilities", paired_probabilities),
        ("interaction-only: final logits", interaction_logits),
        ("interaction-only: final probabilities", interaction_probabilities),
    ):
        mean_values = values.mean(dim=0)
        summary_rows.append({
            "measurement": name,
            "sign_agreement": float((mean_values.sign() == expected).float().mean()),
            "correlation_with_discovery_effect": float(
                torch.corrcoef(torch.stack([predicted, mean_values]))[0, 1]
            ),
        })
    summary = pd.DataFrame(summary_rows)
    decomposition = pd.DataFrame([{
        "source_layer": source_layer,
        "step": step,
        "one_sided_dominance_fraction": float(
            (one_sided_logits.mean(0).abs() > mixed_logits.mean(0).abs()).float().mean()
        ),
        "mean_abs_one_sided_logit": float(one_sided_logits.mean(0).abs().mean()),
        "mean_abs_mixed_logit": float(mixed_logits.mean(0).abs().mean()),
        "one_sided_to_mixed_ratio": float(
            one_sided_logits.mean(0).abs().mean() / mixed_logits.mean(0).abs().mean().clamp_min(1e-30)
        ),
    }])

    colors = results["direction"].map({"promoted": "#24796c", "suppressed": "#b34b42"})
    figure, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    axes[0].scatter(
        results["mean_logit_effect"],
        results["interaction_logit_effect"],
        c=colors,
        alpha=0.85,
    )
    limits = [
        min(results["mean_logit_effect"].min(), results["interaction_logit_effect"].min()),
        max(results["mean_logit_effect"].max(), results["interaction_logit_effect"].max()),
    ]
    axes[0].plot(limits, limits, color="#555555", linestyle="--", linewidth=0.9)
    axes[0].set_xlabel("Discovery mixed-logit effect")
    axes[0].set_ylabel("Held-out mixed-logit effect")
    axes[0].set_title("Interaction reproduces at final logits")

    plotted = pd.concat([
        results[results["direction"] == "suppressed"].iloc[::-1],
        results[results["direction"] == "promoted"],
    ])
    axes[1].barh(
        plotted["token"],
        plotted["paired_probability_delta_ppm"],
        color=colors.loc[plotted.index],
        alpha=0.9,
    )
    axes[1].axvline(0, color="#333333", linewidth=0.8)
    axes[1].set_xlabel("Paired-steer probability change (ppm)")
    axes[1].set_title("Actual final next-token distribution")
    figure.suptitle(
        f"Factor {factor_index} | source layer {source_layer} | "
        f"{num_contexts} held-out FineWeb contexts | step={step}"
    )
    plt.show()

    display(summary)
    display(decomposition)
    display(results[[
        "direction", "rank", "token", "baseline_probability",
        "paired_logit_delta", "mixed_logit_contribution",
        "one_sided_logit_contribution", "paired_probability_delta_ppm",
        "interaction_logit_effect", "interaction_probability_effect",
    ]])
    return {
        "summary": summary,
        "decomposition": decomposition,
        "tokens": results,
        "corners": corners,
    }


final_steering_validation = validate_final_token_steering(
    factor_index=FACTOR_INDEX,
    token_effects=factor_token_effects,
    num_contexts=STEERING_CONTEXTS,
    sequence_length=SCAN_CONFIG["sequence_length"],
    context_batch_size=SCAN_CONFIG["context_batch_size"],
    sample_seed=STEERING_SEED,
    step=SCAN_CONFIG["step"],
)

In [ ]:
# Repeat the behavioral and final-output analysis for the layer 14 -> 18 DCT.
LATE_CONFIG = {
    **SCAN_CONFIG,
    "source_layer": 14,
    "target_layer": 18,
}
LATE_WEIGHTS = project_path + "/sparse-asymmetric/bilinear_attention_pr_l14_l18_weights/seed_0/ov_0.03"
late_L = torch.load(LATE_WEIGHTS + "/L.pt", map_location=device, weights_only=True)
late_R = torch.load(LATE_WEIGHTS + "/R.pt", map_location=device, weights_only=True)
late_U = torch.load(LATE_WEIGHTS + "/U.pt", map_location=device, weights_only=True)

late_behavior_scan = scan_behavior_consistency(
    model=model,
    tokenizer=tokenizer,
    left_factors=late_L,
    right_factors=late_R,
    factor_indices=range(late_L.shape[1]),
    source_layer=LATE_CONFIG["source_layer"],
    num_contexts=LATE_CONFIG["num_contexts"],
    sequence_length=LATE_CONFIG["sequence_length"],
    context_batch_size=LATE_CONFIG["context_batch_size"],
    sample_seed=LATE_CONFIG["sample_seed"],
    shuffle_buffer=LATE_CONFIG["shuffle_buffer"],
    step=LATE_CONFIG["step"],
    top_tokens=LATE_CONFIG["top_tokens"],
    split_repeats=LATE_CONFIG["split_repeats"],
    minimum_effect_quantile=LATE_CONFIG["minimum_effect_quantile"],
    display_count=8,
)
LATE_FACTOR_INDEX = int(late_behavior_scan["ranked"].iloc[0]["factor"])
print(f"Selected late factor: {LATE_FACTOR_INDEX}")

late_factor_token_effects = show_factor_token_effects(
    factor_index=LATE_FACTOR_INDEX,
    tokens_per_direction=TOKENS_PER_DIRECTION,
    num_contexts=LATE_CONFIG["num_contexts"],
    sequence_length=LATE_CONFIG["sequence_length"],
    context_batch_size=LATE_CONFIG["context_batch_size"],
    sample_seed=LATE_CONFIG["sample_seed"],
    left_factors=late_L,
    right_factors=late_R,
    source_layer=LATE_CONFIG["source_layer"],
    step=LATE_CONFIG["step"],
    shuffle_buffer=LATE_CONFIG["shuffle_buffer"],
)
late_final_steering_validation = validate_final_token_steering(
    factor_index=LATE_FACTOR_INDEX,
    token_effects=late_factor_token_effects,
    num_contexts=STEERING_CONTEXTS,
    sequence_length=LATE_CONFIG["sequence_length"],
    context_batch_size=LATE_CONFIG["context_batch_size"],
    sample_seed=STEERING_SEED,
    step=LATE_CONFIG["step"],
    left_factors=late_L,
    right_factors=late_R,
    source_layer=LATE_CONFIG["source_layer"],
    shuffle_buffer=LATE_CONFIG["shuffle_buffer"],
)

slice_comparison = pd.concat([
    final_steering_validation["decomposition"].assign(slice="8 -> 12"),
    late_final_steering_validation["decomposition"].assign(slice="14 -> 18"),
]).set_index("slice")
display(slice_comparison)